In [1]:
!pip -q install opencv-python-headless scikit-image timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.1 MB/s eta 0:00:00


In [2]:
# ============================================
# Stage 1 — ResNet50 + Tiny Transformer + FFT Adapter (Baseline)
# ============================================

import os, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
warnings.filterwarnings("ignore")

DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"


# ---------- Metrics ----------
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc


# ---------- Data ----------
def make_loaders(data_root, batch_size=16):
    size = 320
    train_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    train_ds = datasets.ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    valid_ds = datasets.ImageFolder(os.path.join(data_root, "valid"), transform=eval_tf)
    test_ds  = datasets.ImageFolder(os.path.join(data_root, "test"),  transform=eval_tf)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    fake_idx = train_ds.class_to_idx.get("fake", 0)
    print(f"class_to_idx: {train_ds.class_to_idx}  →  fake_idx={fake_idx}")
    return train_dl, valid_dl, test_dl, fake_idx


In [3]:

# ---------- FFT Adapter ----------
class FFTAdapter(nn.Module):
    """
    Computes 2D FFT on a feature map (B,C,H,W).
    Extracts magnitude and phase, projects each with 1x1 conv,
    concatenates and mixes with 3x3 conv, returns same-channel feature map.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.reduce_mag   = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.reduce_phase = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.mix          = nn.Conv2d(out_ch * 2, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn           = nn.BatchNorm2d(out_ch)
        self.act          = nn.ReLU(inplace=True)

    def forward(self, x):
        X     = torch.fft.fft2(x, dim=(-2, -1))
        mag   = torch.log1p(torch.abs(X))
        phase = torch.angle(X)

        # per-sample standardisation (non-inplace)
        mag   = (mag   - mag.mean(dim=(2, 3), keepdim=True))   / (mag.std(dim=(2, 3), keepdim=True)   + 1e-6)
        phase = (phase - phase.mean(dim=(2, 3), keepdim=True)) / (phase.std(dim=(2, 3), keepdim=True) + 1e-6)

        m = self.reduce_mag(mag)
        p = self.reduce_phase(phase)
        return self.act(self.bn(self.mix(torch.cat([m, p], dim=1))))


# ---------- Tiny Transformer Encoder ----------
class TinyTransformerEncoder(nn.Module):
    def __init__(self, dim=1024, depth=2, heads=8, mlp_ratio=2.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=int(dim * mlp_ratio),
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)

    def forward(self, x):
        return self.enc(x)


# ---------- P3 Model ----------
class Stage1Model(nn.Module):
    def __init__(self, use_fft=False, fft_fuse='concat',
                 freeze_backbone=False, freeze_encoder=False):
        super().__init__()
        self.use_fft  = use_fft
        self.fft_fuse = fft_fuse

        self.backbone = resnet50(weights=None)
        self.backbone.fc = nn.Identity()

        self.neck     = nn.Conv2d(2048, 1024, kernel_size=1, bias=False)
        self.neck_bn  = nn.BatchNorm2d(1024)
        self.neck_act = nn.ReLU(inplace=True)

        self.pool_tokens = nn.AdaptiveAvgPool2d((4, 4))   # 16 tokens
        self.encoder     = TinyTransformerEncoder(dim=1024, depth=2, heads=8, mlp_ratio=2.0)

        if use_fft:
            self.fft = FFTAdapter(in_ch=1024, out_ch=1024)

        head_dim  = 1024 * 2 if (use_fft and fft_fuse == 'concat') else 1024
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(head_dim, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 2)
        )

        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False
            for p in self.neck.parameters():     p.requires_grad = False
            for p in self.neck_bn.parameters():  p.requires_grad = False
        if freeze_encoder:
            for p in self.encoder.parameters():  p.requires_grad = False

    def forward(self, x):
        x = self.backbone.conv1(x);  x = self.backbone.bn1(x)
        x = self.backbone.relu(x);   x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x); x = self.backbone.layer2(x)
        x = self.backbone.layer3(x); x = self.backbone.layer4(x)

        h = self.neck_act(self.neck_bn(self.neck(x)))     # (B, 1024, h, w)

        tokens = self.pool_tokens(h)                       # (B, 1024, 4, 4)
        B, C, H, W = tokens.shape
        tokens = tokens.reshape(B, C, H * W).permute(0, 2, 1)   # (B, 16, 1024)
        z      = self.encoder(tokens)                            # (B, 16, 1024)

        z_map = z.permute(0, 2, 1).reshape(B, C, H, W)
        z_map = F.interpolate(z_map, size=h.shape[-2:], mode='bilinear', align_corners=False)

        if self.use_fft:
            neck_feat = h.detach() if not any(p.requires_grad for p in self.neck.parameters()) else h
            f     = self.fft(neck_feat)
            fused = torch.cat([z_map, f], dim=1) if self.fft_fuse == 'concat' else z_map + f
        else:
            fused = z_map

        return self.head(self.gap(fused))



In [4]:

# ---------- Train / Evaluate ----------
def train(model, dl, optimizer, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss   = ce(logits, y)
        loss.backward()
        optimizer.step()
        b = x.size(0)
        total_loss += loss.item() * b
        total_acc  += (logits.argmax(1) == y).float().sum().item()
        n += b
    return total_loss / n, total_acc / n


@torch.no_grad()
def evaluate(model, dl, device, fake_idx=0):
    model.eval()
    ce = nn.CrossEntropyLoss()
    total_loss, n = 0.0, 0
    y_true, y_prob = [], []
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += ce(logits, y).item() * x.size(0)
        n += x.size(0)
        probs_fake = torch.softmax(logits, dim=1)[:, fake_idx].cpu().numpy()
        y_prob.extend(probs_fake)
        y_true.extend((y.cpu().numpy() == fake_idx).astype(int))
    auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
    return total_loss / n, auc, f1, eer, acc




In [5]:
# ---------- Checkpoint ----------
def save_ckpt(model, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model": model.state_dict()}, path)

def load_ckpt_compatible(model, path):
    ckpt = torch.load(path, map_location="cpu")["model"]
    msd  = model.state_dict()
    keep    = {k: v for k, v in ckpt.items() if k in msd and v.shape == msd[k].shape}
    skipped = len(ckpt) - len(keep)
    model.load_state_dict(keep, strict=False)
    print(f"Loaded {len(keep)} tensors; skipped {skipped} (shape mismatch).")


# ============================================
# Run A — Baseline (no FFT)
# ============================================
def run(use_fft, save_path, ckpt_path=None,
        epochs=5, batch_size=32, lr=1e-3,
        freeze_backbone=True, freeze_encoder=True):

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_dl, valid_dl, test_dl, fake_idx = make_loaders(DATA_ROOT, batch_size)

    model = Stage1Model(
        use_fft=use_fft, fft_fuse='concat',
        freeze_backbone=freeze_backbone,
        freeze_encoder=freeze_encoder
    ).to(device)

    if ckpt_path and os.path.isfile(ckpt_path):
        print(f"Loading compatible weights from {ckpt_path}")
        load_ckpt_compatible(model, ckpt_path)

    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    best_auc = 0.0
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc              = train(model, train_dl, opt, device)
        va_loss, auc, f1, eer, acc   = evaluate(model, valid_dl, device, fake_idx)
        print(
            f"Epoch {ep:02d}: train loss={tr_loss:.4f} acc={tr_acc:.4f} | "
            f"val: loss={va_loss:.4f} | AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}"
        )
        if auc > best_auc:
            best_auc = auc
            save_ckpt(model, save_path)
            print(f"  ↳ Saved best to {save_path}")

    print("\nLoading best checkpoint for test evaluation…")
    load_ckpt_compatible(model, save_path)
    te_loss, auc, f1, eer, acc = evaluate(model, test_dl, device, fake_idx)
    print(
        f"TEST → loss={te_loss:.4f} | "
        f"AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}"
    )
    return model




In [6]:
print("\n" + "="*50)
print("Stage 1A — Baseline (frozen backbone, no FFT)")
print("="*50)
model_base = run(
    use_fft=False,
    save_path='/kaggle/working/p3_stage1_base.pth',
    epochs=1, batch_size=32, lr=1e-3,
    freeze_backbone=True, freeze_encoder=True
)

print("\n" + "="*50)
print("Stage 1B — FFT Adapter (frozen backbone + FFT branch)")
print("="*50)
model_fft = run(
    use_fft=True,
    save_path='/kaggle/working/p3_stage1_fft.pth',
    ckpt_path='/kaggle/working/p3_stage1_base.pth',
    epochs=2, batch_size=32, lr=5e-4,
    freeze_backbone=True, freeze_encoder=True
)


Stage 1A — Baseline (frozen backbone, no FFT)
class_to_idx: {'fake': 0, 'real': 1}  →  fake_idx=0
Epoch 01: train loss=0.6764 acc=0.5790 | val: loss=0.6712 | AUROC=0.635 | F1=0.653 | EER=0.404 | ACC=0.591
  ↳ Saved best to /kaggle/working/p3_stage1_base.pth

Loading best checkpoint for test evaluation…
Loaded 352 tensors; skipped 0 (shape mismatch).
TEST → loss=0.6727 | AUROC=0.631 | F1=0.647 | EER=0.406 | ACC=0.585

Stage 1B — FFT Adapter (frozen backbone + FFT branch)
class_to_idx: {'fake': 0, 'real': 1}  →  fake_idx=0
Loading compatible weights from /kaggle/working/p3_stage1_base.pth
Loaded 351 tensors; skipped 1 (shape mismatch).
Epoch 01: train loss=0.6656 acc=0.5975 | val: loss=0.6480 | AUROC=0.674 | F1=0.653 | EER=0.372 | ACC=0.624
  ↳ Saved best to /kaggle/working/p3_stage1_fft.pth
Epoch 02: train loss=0.6514 acc=0.6193 | val: loss=0.6438 | AUROC=0.678 | F1=0.647 | EER=0.370 | ACC=0.628
  ↳ Saved best to /kaggle/working/p3_stage1_fft.pth

Loading best checkpoint for test evalu

## Optional: Skip Training and Evaluate a Provided Checkpoint

If you do not want to rerun training, you may instead upload the provided pretrained `.pth` checkpoint as a Kaggle dataset and evaluate it directly.

In that case:
- do not run the training cells above(comment them out)
- update the checkpoint path in the evaluation-only cell below(uncomment cells below)
- run only the evaluation-only cell

This allows reproduction of the final test result without retraining the model from scratch.


In [ ]:
# # ============================================
# # Optional evaluation-only cell for Stage 1A
# # ============================================

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# _, _, test_dl, fake_idx = make_loaders(DATA_ROOT, batch_size=32)

# model = Stage1Model(
#     use_fft=False,
#     fft_fuse='concat',
#     freeze_backbone=True,
#     freeze_encoder=True
# ).to(device)

# CKPT_PATH = "/kaggle/input/YOUR_DATASET_NAME/p3_stage1_base.pth"   # change this
# load_ckpt_compatible(model, CKPT_PATH)

# te_loss, auc, f1, eer, acc = evaluate(model, test_dl, device, fake_idx)
# print(f"TEST ONLY → loss={te_loss:.4f} | AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")


In [ ]:
# # ============================================
# # Optional evaluation-only cell for Stage 1B
# # ============================================

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# _, _, test_dl, fake_idx = make_loaders(DATA_ROOT, batch_size=32)

# model = Stage1Model(
#     use_fft=True,
#     fft_fuse='concat',
#     freeze_backbone=True,
#     freeze_encoder=True
# ).to(device)

# CKPT_PATH = "/kaggle/input/YOUR_DATASET_NAME/p3_stage1_fft.pth"   # change this
# load_ckpt_compatible(model, CKPT_PATH)

# te_loss, auc, f1, eer, acc = evaluate(model, test_dl, device, fake_idx)
# print(f"TEST ONLY → loss={te_loss:.4f} | AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")
